## Meta Data Aware RAG

In [1]:
from langgraph.graph import StateGraph, START, END, add_messages
from langchain_aws import ChatBedrockConverse, BedrockEmbeddings
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt.tool_node import ToolNode
from langchain_community.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from typing import TypedDict, Annotated, Literal, Optional, Tuple, List
import os
import operator
from rich import print
from langchain.document_loaders import PyPDFLoader

In [2]:
from langchain_aws import ChatBedrockConverse, BedrockEmbeddings

embedding_model = BedrockEmbeddings(model_id="cohere.embed-english-v3")

In [3]:
llm= ChatBedrockConverse(model="cohere.command-r-plus-v1:0", max_tokens=300)

In [4]:
import json
from langchain_core.documents import Document

# Load semi-structured policy data from a JSON file
with open("policy.json", "r") as f:
    data = json.load(f)

# Initialize a list to hold LangChain Document objects
documents = []

# Convert each JSON policy entry into a Document
for item in data:
    documents.append(
        Document(
            # Main textual content used for embedding and semantic search
            page_content=item["content"],
            
            # Metadata preserves structural and descriptive attributes
            # and will later be used for filtering during retrieval
            metadata={
                "id": item["id"],          # Unique policy identifier
                "title": item["title"],    # Policy title for traceability
                "section": item["section"],# Logical category (e.g., Leave, Retention)
                "source": "json"           # Source type for provenance tracking
            }
        )
    )
documents

[Document(metadata={'id': 'P001', 'title': 'Data Privacy Policy', 'section': 'Retention', 'source': 'json'}, page_content='User data is retained for 90 days. It should not be retained after that.'),
 Document(metadata={'id': 'P002', 'title': 'Holiday Policy', 'section': 'Leave', 'source': 'json'}, page_content='Only 10 holidays should be given for any employee in a given year. It includes optional Holidays as well.'),
 Document(metadata={'id': 'P003', 'title': 'Earned Leave Policy', 'section': 'Leave', 'source': 'json'}, page_content="Each employee is eligible for 5 earned leaves per quarter, aligning with our commitment to supporting work–life balance while ensuring smooth operational continuity. Earned leaves are designed to provide team members with the flexibility to pause, rest, or attend to personal commitments without compromising their professional responsibilities. These leaves may be availed at any time during the quarter, subject to prior approval from the reporting manager 

In [5]:
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_core.documents import Document
# # Initialize a recursive text splitter to break large documents
# # into manageable, semantically coherent chunks
# splitter = RecursiveCharacterTextSplitter(
#    chunk_size=300,      # Maximum number of characters per chunk - kept 800 as the largest document to be chunked was properly fitting to this value
#    chunk_overlap=20     # Overlap to preserve context across chunks
# )
# # List to store chunked Document objects
# chunked_docs = []
# # Iterate over each original document
# for doc in documents:
#    # Split the document content into overlapping chunks
#    chunks = splitter.split_text(doc.page_content)
#    print(chunks)
#    # # Create a new Document for each chunk
#    # for i, chunk in enumerate(chunks):
#    #     chunked_docs.append(
#    #         Document(
#    #             # Chunk-level text used for embedding and retrieval
#    #             page_content=chunk,
               
#    #             # Preserve original metadata and add chunk identifier
#    #             # to maintain traceability back to the source document
#    #             metadata={**doc.metadata, "chunk_id": i}
#    #         )
#    #     )


In [6]:
# --------------------------
# Embeddings & Vector Store
# --------------------------
from langchain.vectorstores import Chroma
import chromadb.config
# Create / connect to a Chroma vector store
# Metadata (id, title, section, source, chunk_id) is stored
# alongside each embedding to enable metadata-aware retrieval
vectorstore = Chroma.from_documents(
    documents=documents,          # Chunked documents with metadata
    embedding=embedding_model,
    persist_directory="./chroma_db", # Optional: persist vectors locally
    collection_metadata={
        "description": "Policy documents indexed with metadata-aware embeddings",
        "source_type": "json"
    }
)

In [7]:
# Convert the vector store into a retriever interface
# This retriever will be used by the RAG pipeline to fetch relevant context
vectorstore_db = vectorstore.as_retriever(
    
    # Use Maximal Marginal Relevance (MMR) for retrieval
    # MMR balances relevance with diversity to avoid redundant chunks
    search_type="mmr",
    
    search_kwargs={
        "k": 3,                         # Number of chunks to retrieve
        "filter": {"section": "Leave"}, # Metadata-based filter to scope retrieval
                                          # Ensures only 'Leave' related policies are considered
        "lambda_mult": 0.3              # Controls relevance vs diversity trade-off
                                          # Lower value increases diversity
    }
)

In [8]:
from langchain_core.prompts import ChatPromptTemplate

# Define a chat prompt template for the RAG pipeline
# The prompt strictly constrains the LLM to use only retrieved context
prompt = ChatPromptTemplate.from_messages([
    
    # --------------------
    # System Instruction
    # --------------------
    (
        "system",
        """
You are a policy assistant.

Answer user questions strictly based on the provided context.
Do not introduce external knowledge or assumptions.

<context>
{context}
</context>
"""
    ),

    # --------------------
    # Human Query
    # --------------------
    (
        "human",
        "{input}"   # Placeholder for the user's actual question
    )
])

# Print the final prompt structure for verification/debugging
print(prompt)

ChatPromptTemplate(
    input_variables=['context', 'input'],
    input_types={},
    partial_variables={},
    messages=[
        SystemMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=['context'],
                input_types={},
                partial_variables={},
                template='\nYou are a policy assistant.\n\nAnswer user questions strictly based on the provided 
context.\nDo not introduce external knowledge or assumptions.\n\n<context>\n{context}\n</context>\n'
            ),
            additional_kwargs={}
        ),
        HumanMessagePromptTemplate(
            prompt=PromptTemplate(
                input_variables=['input'],
                input_types={},
                partial_variables={},
                template='{input}'
            ),
            additional_kwargs={}
        )
    ]
)

In [9]:
from langchain_core.prompts import PromptTemplate

# Define a document-level prompt template
# This template formats each retrieved document chunk Chunk ID:{chunk_id}
# before it is passed to the language model as context
document_prompt = PromptTemplate.from_template(
    """
Source: {source}
Policy ID: {id}
Section: {section}


Content:
{page_content}
"""
)

# The placeholders (source, id, section, page_content)
# are automatically populated from each Document's metadata
# and text, ensuring traceable and well-structured context

In [10]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# Create a document combination chain
# This chain "stuffs" all retrieved document chunks into the prompt
# using the previously defined document_prompt and main chat prompt
docs_chain = create_stuff_documents_chain(
    llm=llm,                 # Language model used for generation
    prompt=prompt,           # Chat-level prompt with system and human messages
    document_prompt=document_prompt,  # Formats each retrieved document
    document_variable_name="context"
)

# Create the retrieval chain
# This chain links the retriever with the document combination chain
# It first retrieves metadata-filtered, semantically relevant chunks
# and then passes them to the LLM for grounded answer generation
rag_chain = create_retrieval_chain(
    vectorstore_db,  # Metadata-aware retriever (MMR + filters)
    docs_chain
)


In [11]:
print(rag_chain.invoke({"input":"what does policy says aboout unused earned leaves ?"}))

{
    'input': 'what does policy says aboout unused earned leaves ?',
    'context': [
        Document(
            metadata={'title': 'Earned Leave Policy', 'section': 'Leave', 'source': 'json', 'id': 'P003'},
            page_content="Each employee is eligible for 5 earned leaves per quarter, aligning with our commitment 
to supporting work–life balance while ensuring smooth operational continuity. Earned leaves are designed to provide
team members with the flexibility to pause, rest, or attend to personal commitments without compromising their 
professional responsibilities. These leaves may be availed at any time during the quarter, subject to prior 
approval from the reporting manager to ensure adequate workforce planning.For trainees, the entitlement is 4 earned
leaves per quarter. This distinction recognizes the structured nature of training programs and the importance of 
consistent participation during the learning phase. Trainees are encouraged to utilize these leaves responsibly and
plan ahead whenever possible. All employees and trainees are expected to submit leave requests through the 
designated HR or attendance system. Unused earned leaves will follow the organization's standard carry‑forward or 
lapse rules as applicable."
        ),
        Document(
            metadata={'source': 'json', 'title': 'Earned Leave Policy', 'section': 'Leave', 'id': 'P003'},
            page_content="Each employee is eligible for 5 earned leaves per quarter, aligning with our commitment 
to supporting work–life balance while ensuring smooth operational continuity. Earned leaves are designed to provide
team members with the flexibility to pause, rest, or attend to personal commitments without compromising their 
professional responsibilities. These leaves may be availed at any time during the quarter, subject to prior 
approval from the reporting manager to ensure adequate workforce planning.For trainees, the entitlement is 4 earned
leaves per quarter. This distinction recognizes the structured nature of training programs and the importance of 
consistent participation during the learning phase. Trainees are encouraged to utilize these leaves responsibly and
plan ahead whenever possible. All employees and trainees are expected to submit leave requests through the 
designated HR or attendance system. Unused earned leaves will follow the organization's standard carry‑forward or 
lapse rules as applicable."
        ),
        Document(
            metadata={'id': 'P002', 'source': 'json', 'title': 'Holiday Policy', 'section': 'Leave'},
            page_content='Only 10 holidays should be given for any employee in a given year. It includes optional 
Holidays as well.'
        )
    ],
    'answer': "Unused earned leaves will follow the organization's standard carry-forward or lapse rules."
}

In [12]:
print(rag_chain.invoke({"input":"what are the id's related to earned leave and holiday policy. Mention all.?"}))

{
    'input': "what are the id's related to earned leave and holiday policy. Mention all.?",
    'context': [
        Document(
            metadata={'section': 'Leave', 'id': 'P003', 'source': 'json', 'title': 'Earned Leave Policy'},
            page_content="Each employee is eligible for 5 earned leaves per quarter, aligning with our commitment 
to supporting work–life balance while ensuring smooth operational continuity. Earned leaves are designed to provide
team members with the flexibility to pause, rest, or attend to personal commitments without compromising their 
professional responsibilities. These leaves may be availed at any time during the quarter, subject to prior 
approval from the reporting manager to ensure adequate workforce planning.For trainees, the entitlement is 4 earned
leaves per quarter. This distinction recognizes the structured nature of training programs and the importance of 
consistent participation during the learning phase. Trainees are encouraged to utilize these leaves responsibly and
plan ahead whenever possible. All employees and trainees are expected to submit leave requests through the 
designated HR or attendance system. Unused earned leaves will follow the organization's standard carry‑forward or 
lapse rules as applicable."
        ),
        Document(
            metadata={'section': 'Leave', 'source': 'json', 'title': 'Earned Leave Policy', 'id': 'P003'},
            page_content="Each employee is eligible for 5 earned leaves per quarter, aligning with our commitment 
to supporting work–life balance while ensuring smooth operational continuity. Earned leaves are designed to provide
team members with the flexibility to pause, rest, or attend to personal commitments without compromising their 
professional responsibilities. These leaves may be availed at any time during the quarter, subject to prior 
approval from the reporting manager to ensure adequate workforce planning.For trainees, the entitlement is 4 earned
leaves per quarter. This distinction recognizes the structured nature of training programs and the importance of 
consistent participation during the learning phase. Trainees are encouraged to utilize these leaves responsibly and
plan ahead whenever possible. All employees and trainees are expected to submit leave requests through the 
designated HR or attendance system. Unused earned leaves will follow the organization's standard carry‑forward or 
lapse rules as applicable."
        ),
        Document(
            metadata={'section': 'Leave', 'id': 'P002', 'source': 'json', 'title': 'Holiday Policy'},
            page_content='Only 10 holidays should be given for any employee in a given year. It includes optional 
Holidays as well.'
        )
    ],
    'answer': 'Policy ID: P003 (Earned Leave Policy) and Policy ID: P002 (Holiday Policy)'
}